# Lab 03 — Classic NLP: Text Preprocessing & Sentiment Analysis

## Business Context

**ShopSmart** has been collecting customer reviews for months. The customer success team has one question:

> *"Can we automatically classify reviews as positive or negative, and understand what topics customers mention most?"*

We'll solve this with **classic NLP** — no neural networks, no APIs. Just text processing + ML.

---

## Learning Objectives

- Understand the NLP pipeline: raw text → features → model
- Apply text preprocessing: lowercasing, punctuation removal, stopwords, stemming/lemmatisation
- Convert text to numbers with **Bag of Words** and **TF-IDF**
- Train a sentiment classifier (Logistic Regression on TF-IDF)
- Explore top terms per class and build a word cloud

---

## Setup

In [ ]:
# Install dependencies if needed
# !pip install nltk wordcloud scikit-learn pandas matplotlib seaborn

In [ ]:
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from wordcloud import WordCloud

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

sns.set_theme(style='whitegrid')
print('Ready!')

---

## Part 1 — The Dataset: Customer Reviews

In [ ]:
reviews_data = [
    # Positive reviews (label = 1)
    ("Absolutely love this product! Delivery was super fast and packaging was perfect.", 1),
    ("Great quality, exactly as described. Will definitely buy again from ShopSmart.", 1),
    ("The customer service team was incredibly helpful and resolved my issue quickly.", 1),
    ("Amazing value for money. The product exceeded my expectations!", 1),
    ("Fast shipping and the item looks even better in person. Very satisfied!", 1),
    ("Smooth checkout process and arrived two days earlier than expected. Love it!", 1),
    ("Top quality product. My family is very happy with the purchase.", 1),
    ("Easy to use, well designed and the price is unbeatable. Highly recommended.", 1),
    ("Ordered twice already. Reliable service, quality products, fast delivery.", 1),
    ("The product works perfectly. Came well packed, no damage. 5 stars!", 1),
    ("Excellent experience from start to finish. Support was friendly and fast.", 1),
    ("Really impressed with the build quality. Way better than competitors.", 1),
    ("Happy customer here! Got a discount code too — great service overall.", 1),
    ("Product is solid, instructions clear. Installation was simple.", 1),
    ("Very responsive shop. Had a question and got an answer within minutes.", 1),
    ("Gorgeous item, premium feel. Makes a wonderful gift.", 1),
    ("Accurate description, speedy delivery, great price. Nothing to complain about.", 1),
    ("Bought this as a gift and the recipient was thrilled. Thank you ShopSmart!", 1),
    ("Durable and functional. Exactly what I needed for my home office.", 1),
    ("Best purchase I made this year. Already recommended it to three friends.", 1),
    # Negative reviews (label = 0)
    ("Terrible experience. Product broke after two days. Very disappointed.", 0),
    ("Delivery took 3 weeks and the item arrived damaged. Unacceptable!", 0),
    ("Completely different from the photos. Misleading description. Avoid.", 0),
    ("Customer support was rude and unhelpful. Will never shop here again.", 0),
    ("Cheap quality, not worth the price at all. Waste of money.", 0),
    ("The product stopped working after one week. No response from support.", 0),
    ("Very disappointed. Packaging was a mess and parts were missing.", 0),
    ("Worst online purchase I ever made. Refund process is a nightmare.", 0),
    ("Size was totally wrong despite following the guide. Frustrated.", 0),
    ("Took months to arrive, no tracking updates, and it was the wrong item.", 0),
    ("Poor build quality and it smells bad. Not as advertised at all.", 0),
    ("Scam! I paid for express delivery and it still took two weeks.", 0),
    ("Instructions are useless. Product is very hard to assemble.", 0),
    ("Broken on arrival. Support took 10 days to reply and offered no solution.", 0),
    ("Do not buy. Cheap material, looks nothing like the pictures.", 0),
    ("Wrong colour delivered, shop refused to replace. Shocking service.", 0),
    ("The product is a total disappointment. Flimsy and poorly made.", 0),
    ("Late delivery, broken packaging, and missing accessories. Terrible.", 0),
    ("Ordered two months ago, still not received. Support ignores my emails.", 0),
    ("Zero quality control. Item clearly used and re-packaged as new.", 0),
]

df = pd.DataFrame(reviews_data, columns=['review', 'sentiment'])
print(df['sentiment'].value_counts())
df.head()

---

## Part 2 — Text Preprocessing

### 2.1 The NLP Pipeline

```
Raw text
  → Lowercase
  → Remove punctuation & digits
  → Tokenise (split into words)
  → Remove stopwords ("the", "is", "a", ...)
  → Stem or Lemmatise
  → Clean token list
```

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text: str) -> str:
    # 1. Lowercase
    text = text.lower()
    # 2. Remove punctuation and digits
    text = re.sub(r'[^a-z\s]', '', text)
    # 3. Tokenise
    tokens = word_tokenize(text)
    # 4. Remove stopwords and short tokens
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    # 5. Lemmatise
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df['clean_review'] = df['review'].apply(preprocess)

# Compare raw vs clean
for i in range(3):
    print(f'RAW:   {df["review"][i]}')
    print(f'CLEAN: {df["clean_review"][i]}')
    print()

---

## Part 3 — Text to Numbers

### 3.1 Bag of Words (CountVectorizer)

Each review becomes a vector where each dimension represents a word and the value is its count.

In [ ]:
bow_vectorizer = CountVectorizer(max_features=50)
X_bow = bow_vectorizer.fit_transform(df['clean_review'])

bow_df = pd.DataFrame(X_bow.toarray(), columns=bow_vectorizer.get_feature_names_out())
print('Shape:', bow_df.shape)
bow_df.head(3)

### 3.2 TF-IDF — Term Frequency × Inverse Document Frequency

> **Intuition:** A word is important in a document if it appears often there, *and* it is rare across all documents.
> - TF: how often the word appears in this document
> - IDF: penalises words that appear in every document (they carry little meaning)

In [ ]:
tfidf = TfidfVectorizer(max_features=100, ngram_range=(1, 2))  # unigrams + bigrams
X_tfidf = tfidf.fit_transform(df['clean_review'])

print('TF-IDF matrix shape:', X_tfidf.shape)

### 3.3 Most Informative Terms per Sentiment

In [ ]:
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())
tfidf_df['sentiment'] = df['sentiment'].values

top_n = 15
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, color, title in zip(
    axes,
    [1, 0],
    ['steelblue', 'tomato'],
    ['Top terms — Positive reviews', 'Top terms — Negative reviews']
):
    top_terms = (
        tfidf_df[tfidf_df['sentiment'] == label]
        .drop(columns='sentiment')
        .mean()
        .sort_values(ascending=False)
        .head(top_n)
    )
    ax.barh(top_terms.index[::-1], top_terms.values[::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Avg TF-IDF score')

plt.tight_layout()
plt.show()

### 3.4 Word Cloud

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, label, title in zip(axes, [1, 0], ['Positive Reviews', 'Negative Reviews']):
    text = ' '.join(df[df['sentiment'] == label]['clean_review'])
    wc = WordCloud(width=600, height=300, background_color='white',
                   colormap='Blues' if label == 1 else 'Reds').generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title)

plt.tight_layout()
plt.show()

---

## Part 4 — Sentiment Classification

### 4.1 Train / Test Split

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_review'], df['sentiment'],
    test_size=0.25, random_state=42, stratify=df['sentiment']
)

# Fit TF-IDF ONLY on train, transform both sets
tfidf_model = TfidfVectorizer(max_features=200, ngram_range=(1, 2))
X_train = tfidf_model.fit_transform(X_train_raw)
X_test  = tfidf_model.transform(X_test_raw)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

### 4.2 Train — Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Positive']))

### 4.3 Train — Naïve Bayes (the classic NLP model)

In [ ]:
nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

print('=== Naïve Bayes ===')
print(classification_report(y_test, y_pred_nb, target_names=['Negative', 'Positive']))

### 4.4 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, preds, title in zip(axes,
                             [y_pred_lr, y_pred_nb],
                             ['Logistic Regression', 'Naïve Bayes']):
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=['Negative', 'Positive']).plot(ax=ax, colorbar=False)
    ax.set_title(title)
plt.tight_layout()
plt.show()

### 4.5 Predict on New Text

In [ ]:
new_reviews = [
    "The product quality is outstanding and arrived ahead of schedule!",
    "Horrible. Broken on arrival and support never replied to my emails.",
    "It's okay, nothing special but does the job."
]

new_reviews_clean = [preprocess(r) for r in new_reviews]
X_new = tfidf_model.transform(new_reviews_clean)
predictions = lr.predict(X_new)
probabilities = lr.predict_proba(X_new)

for review, pred, prob in zip(new_reviews, predictions, probabilities):
    label = 'POSITIVE' if pred == 1 else 'NEGATIVE'
    confidence = max(prob)
    print(f'[{label} | {confidence:.0%}] {review}')

---

## Your Turn! 🧪

**Exercise A:**
Add 5 more reviews to the dataset (a mix of positive and negative). Re-run the pipeline. Does anything change?

**Exercise B:**
In `TfidfVectorizer`, change `ngram_range=(1, 2)` to `(1, 1)` (unigrams only). Then try `(2, 2)` (bigrams only). How do the results compare?

**Exercise C:**
Modify the `preprocess` function to also remove words shorter than 3 characters instead of 2. Does it help? Why or why not?

**Exercise D (Bonus):**
Remove the lemmatization step from `preprocess`. What happens to the vocabulary size and model performance?

In [ ]:
# Your code here


---

## Summary

```
Raw text → Preprocessing → Vectorisation → ML model → Prediction
```

| Step | Tools |
|---|---|
| Preprocessing | NLTK: tokenise, stopwords, lemmatise |
| Vectorisation | CountVectorizer (BoW), TfidfVectorizer |
| Model | Logistic Regression, Naïve Bayes |
| Evaluation | Precision, Recall, F1, Confusion Matrix |

### Limitations of classic NLP
- No understanding of context ("not bad" is hard to classify correctly)
- Requires manual feature engineering
- Struggles with sarcasm, idioms, domain-specific language

**Next session:** We'll use **Generative AI** to overcome these limitations with large language models.